In [63]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import BaseOutputParser

chat = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5.1",
    temperature=0.1,
    # streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

trans_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a translator."),
        ("human", "translate {source_lang} to {target_lang} text: {text}"),
    ]
)

class OutputParser(BaseOutputParser):
    def parse(self, text):
        return text.strip()
    
trans_chain = trans_prompt | chat | OutputParser()

response = trans_chain.invoke({"text": "There are many beautiful places in the world.", "source_lang": "English", "target_lang": "Korean"})


In [59]:
from langchain_openai import ChatOpenAI
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_core.prompts import ChatPromptTemplate


chat = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5.1",
    # temperature=0.1,   
    # streaming=True,
    callbacks= [StreamingStdOutCallbackHandler()]
)

sum_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system","Summarize the following topic in 3 sentences",
        ),
        ("human", "{topic}"),
    ]
)

quiz_prompt = ChatPromptTemplate.from_messages(
    [     
        ("system", "create only 1 short question maximum 10 words."),
        ("human", "Based on {summary}"),
    ]
)

class OutputParser(BaseOutputParser):
    def parse(self, text):
        return text.strip()
    

quiz_chain = quiz_prompt | chat | OutputParser()

summary_chain = sum_prompt | chat | OutputParser()
    
final_chain = {"summary": summary_chain} | quiz_chain

response = final_chain.invoke({"topic": "The sky is blue."})
response

'Why does Rayleigh scattering affect shorter wavelengths like blue most?'